# 3. Hyperparameter Optimization using Optuna
We will take the best performing model (likely XGBoost) and tune its hyperparameters to squeeze out maximum performance for our CV metric points.


In [ ]:
import os
import sys
import optuna
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, accuracy_score

sys.path.append(os.path.abspath('..'))
from src import config

optuna.logging.set_verbosity(optuna.logging.WARNING)


## Load Data


In [ ]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / 'features.csv')
X = df[config.FEATURE_NAMES]
y_reg = df['CI']
y_cls = df['Label']

X_train, X_test, y_reg_train, y_reg_test = train_test_split(X, y_reg, test_size=0.2, random_state=config.SEED)
X_train_c, X_test_c, y_cls_train, y_cls_test = train_test_split(X, y_cls, test_size=0.2, random_state=config.SEED, stratify=y_cls)


## XGBoost Regressor Optimization (Chatter Index)


In [ ]:
def objective_reg(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': config.SEED
    }
    
    model = XGBRegressor(**params)
    score = cross_val_score(model, X_train, y_reg_train, cv=5, scoring='r2').mean()
    return score

study_reg = optuna.create_study(direction='maximize')
study_reg.optimize(objective_reg, n_trials=50)

print(f"Best R2 Score (CV): {study_reg.best_value:.4f}")
print("Best Params:", study_reg.best_params)

# Train on full train and eval on test
best_reg = XGBRegressor(**study_reg.best_params, random_state=config.SEED)
best_reg.fit(X_train, y_reg_train)
final_r2 = r2_score(y_reg_test, best_reg.predict(X_test))
print(f"\nFinal Test R2 Score: {final_r2:.4f}")


## XGBoost Classifier Optimization (Stable vs Chatter)


In [ ]:
def objective_cls(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': config.SEED,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    score = cross_val_score(model, X_train_c, y_cls_train, cv=5, scoring='accuracy').mean()
    return score

study_cls = optuna.create_study(direction='maximize')
study_cls.optimize(objective_cls, n_trials=50)

print(f"Best Accuracy Score (CV): {study_cls.best_value:.4f}")
print("Best Params:", study_cls.best_params)

best_cls = XGBClassifier(**study_cls.best_params, random_state=config.SEED, eval_metric='logloss')
best_cls.fit(X_train_c, y_cls_train)
final_acc = accuracy_score(y_cls_test, best_cls.predict(X_test_c))
print(f"\nFinal Test Accuracy: {final_acc:.4f}")
